# metadata 분석

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data")

metadata_csv_path = DATA_DIR / "metadata_v2.0.csv"
wonkim_csv_path = DATA_DIR / "wonkim_seg_data.csv"

# 1. CSV 파일 읽기
df_meta = pd.read_csv(metadata_csv_path)
df_wonkim = pd.read_csv(wonkim_csv_path)

df_meta.head(5)

,video_path,common_path,n_frames,n_json,frames_done,sapiens_done,reextract_done,overlay_done,sam_done,id_done,is_train,is_val
0,/workspace/nas203/ds_RehabilitationMedicineDat...,AI_dataset/N01/N01_Treatment/diagonal__biceps_...,2432,2432,True,True,True,True,True,True,False,False
1,/workspace/nas203/ds_RehabilitationMedicineDat...,AI_dataset/N01/N01_Treatment/diagonal__hip_ext...,320,320,True,True,True,True,True,True,True,False
2,/workspace/nas203/ds_RehabilitationMedicineDat...,AI_dataset/N01/N01_Treatment/diagonal__hip_fle...,332,332,True,True,True,True,True,True,True,False
3,/workspace/nas203/ds_RehabilitationMedicineDat...,AI_dataset/N01/N01_Treatment/diagonal__hip_fle...,346,346,True,True,True,True,True,True,True,False
4,/workspace/nas203/ds_RehabilitationMedicineDat...,AI_dataset/N01/N01_Treatment/diagonal__hip_fle...,347,347,True,True,True,True,True,True,True,False


## 

## False인 COMMON PATH 확인

In [40]:
from pathlib import Path
import pandas as pd

# 1. 파일 경로 설정
DATA_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data")
metadata_csv_path = DATA_DIR / "metadata_v2.0.csv"
output_txt_path = DATA_DIR / "unused_list.txt"  # 저장할 텍스트 파일 경로

# 2. 데이터 읽기
df_meta = pd.read_csv(metadata_csv_path)

# 3. is_train과 is_val이 모두 False인 행 필터링
df_unused = df_meta[
    (df_meta['is_train'] == False) & 
    (df_meta['is_val'] == False)
]

print(f"🔍 전체 {len(df_meta)}개 중 미사용 데이터 {len(df_unused)}개를 추출했습니다.")

# 4. 텍스트 파일로 저장 (인덱스 포함, 보기 좋은 포맷)
with open(output_txt_path, "w", encoding="utf-8") as f:
    # 헤더 작성
    f.write(f"Total Count: {len(df_unused)}\n")
    f.write("=" * 80 + "\n")
    f.write(f"{'Index':<8} | {'Common Path'}\n")
    f.write("-" * 80 + "\n")
    
    # 내용 작성 (인덱스와 경로)
    for idx, row in df_unused.iterrows():
        f.write(f"{idx:<8} | {row['common_path']}\n")

print(f"✅ 리스트 저장 완료: {output_txt_path}")

# 5. 저장된 내용 확인 (상위 10줄만 미리보기)
print("\n--- 파일 내용 미리보기 ---")
with open(output_txt_path, "r", encoding="utf-8") as f:
    for _ in range(10):
        print(f.readline().strip())

🔍 전체 1152개 중 미사용 데이터 645개를 추출했습니다.
✅ 리스트 저장 완료: /workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/unused_list.txt

--- 파일 내용 미리보기 ---
Total Count: 645
Index    | Common Path
--------------------------------------------------------------------------------
0        | AI_dataset/N01/N01_Treatment/diagonal__biceps_curl
13       | AI_dataset/N01/N01_Treatment/frontal__hip_flexion_extension_bobath_table
19       | AI_dataset/N01/N01_Treatment/frontal__slr_bobath_table
31       | AI_dataset/N01/N01_Ward/diagonal__biceps_curl
43       | AI_dataset/N01/N01_Ward/diagonal__lumbar_rotation
47       | AI_dataset/N01/N01_Ward/diagonal__pec_dec_fly


## is_train 변경하기

In [35]:
target_idx = 29

print(f"common path : {df_meta.loc[target_idx, 'common_path']}")
print(f"현재 상태: {df_meta.loc[target_idx, 'is_train']}")

# 현재 값의 반대(!not)로 덮어쓰기
df_meta.loc[target_idx, 'is_train'] = not df_meta.loc[target_idx, 'is_train']

print(f"반전 후: {df_meta.loc[target_idx, 'is_train']}")

df_meta.to_csv(metadata_csv_path, index=False)
print(f"💾 파일 저장 완료")

common path : AI_dataset/N01/N01_Treatment/lateral__slr_bobath_table
현재 상태: False
반전 후: True
💾 파일 저장 완료


# 수정하기

### ID 하나만 남기기

In [9]:
import sys
import pandas as pd
from pathlib import Path

# 경로 설정 (사용자 환경에 맞게 수정)
BASE_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/ASAN_01_Repetition_Counter_Final")
sys.path.append(str(BASE_DIR))

# 모듈 불러오기 (경로가 맞는지 확인 필요)
try:
    from utils.post_filtering import filter_and_save_json
except ImportError:
    # 만약 utils 패키지가 아니라면, 함수가 정의된 파일 경로를 확인해야 합니다.
    # 예: from func.filter import filter_and_save_json
    print("❌ 모듈을 찾을 수 없습니다. 경로를 확인해주세요.")

DATA_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data")
metadata_csv_path = DATA_DIR / "metadata_v2.0.csv"

# 데이터 로드
df = pd.read_csv(metadata_csv_path)
target_idx = 47

# 경로 구성
common_path = df.loc[target_idx, "common_path"]
kpt_dir = DATA_DIR / "2_KEYPOINTS" / common_path

# 🛠️ 함수 호출 수정
filter_and_save_json(
    src_kpt_dir=str(kpt_dir),  # 🔴 수정됨: src_kpt_data -> src_kpt_dir
    dst_kpt_dir=str(kpt_dir),  # 원본 덮어쓰기 (주의: 백업 권장)
    target_ids=[2]             # 🔴 수정됨: 2 -> [2] (리스트 형태여야 함)
)


running Filtering... (Target IDs: [2])


Filtering JSON: 100% 419/419 [00:12<00:00, 32.80it/s]

✅ 필터링 완료: 419개의 파일이 'diagonal__pec_dec_fly'에 저장됨.


True

## video 만들기

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/ASAN_01_Repetition_Counter_Final")
DATA_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data")
CSV_PATH = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data/metadata_v2.0.csv")

import sys
sys.path.append(str(BASE_DIR))
from utils.generate_skeleton_video import generate_17kpt_skeleton_video, generate_sam_video

df=pd.read_csv(CSV_PATH)

target = 175
common_path = df.loc[target,"common_path"]
frame_path = DATA_DIR / "1_FRAME" / common_path
kpt_path = DATA_DIR / "2_KEYPOINTS" / common_path
sam_path = DATA_DIR / "8_SAM" / common_path
out_path = DATA_DIR / "test" / f"{common_path}/samtest.mp4"

# generate_17kpt_skeleton_video(
#     frame_dir=frame_path,
#     kpt_dir=kpt_path,
#     output_path=out_path,
#     conf_threshold=0
# )

generate_sam_video(
    frame_dir=frame_path,
    json_dir=sam_path,
    output_path=out_path,
    fps=30.0,  # FPS 설정
    alpha=0.8
)

TypeError: generate_sam_video() got an unexpected keyword argument 'sam_dir'

## 전체 데이터 / 학습 데이터 / 검증 데이터 양 비교

In [16]:
# 요약 데이터 생성
summary_data = {
    "구분": ["전체 데이터", "Train (학습용)", "Val (검증용)"],
    "개수": [len(df_meta), df_meta['is_train'].sum(), df_meta['is_val'].sum()]
}

# 데이터프레임으로 변환
df_summary = pd.DataFrame(summary_data)

# 출력
print("📊 v1.0 데이터셋 할당 현황")
print("=" * 30)
print(df_summary.to_string(index=False))
print("=" * 30)

📊 v1.0 데이터셋 할당 현황
         구분   개수
     전체 데이터 1152
Train (학습용)   12
  Val (검증용)    0


# 보산진 데이터 분석

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/workspace/nas203/ds_RehabilitationMedicineData/IDs/tojihoo/data")
metadata_csv_path = DATA_DIR / "metadata.csv"
wonkim_csv_path = DATA_DIR / "wonkim_seg_data.csv"

# 1. CSV 파일 읽기
df_meta = pd.read_csv(metadata_csv_path)
df_wonkim = pd.read_csv(wonkim_csv_path)

df_wonkim.head()

,common_path,raw_label,start_frame,end_frame,duration
0,Won_Kim_research_at_Bosanjin/M01/M01_VISIT12,frontal__pec_dec_fly__1,150,1585,1435
1,Won_Kim_research_at_Bosanjin/M01/M01_VISIT12,frontal__pec_dec_fly__2,2870,3462,592
2,Won_Kim_research_at_Bosanjin/M01/M01_VISIT12,frontal__shoulder_press__1,4100,5300,1200
3,Won_Kim_research_at_Bosanjin/M01/M01_VISIT12,frontal__shoulder_press__2,5750,7050,1300
4,Won_Kim_research_at_Bosanjin/M01/M01_VISIT12,frontal__shoulder_flexion__1,7400,9400,2000


In [3]:
# 'raw_label'을 '__'로 나누고 중간에 위치한 운동명(index 1)만 추출하여 중복을 제거한 개수를 출력합니다.
print(f"운동 종류 개수: {df_wonkim['raw_label'].str.split('__').str[1].nunique()}개")

print(df_wonkim['raw_label'].str.split('__').str[1].unique())

운동 종류 개수: 32개
['pec_dec_fly' 'shoulder_press' 'shoulder_flexion' 'rowing' 'clamshell'
 'hip_knee_flexion' 'slr' 'leg_cycle_exercise' 'biceps_curl' 'chest_press'
 'longsitting_knee_flexion' 'hooklying_hip_flexion' 'leg_swing'
 'q_setting' 'upright_row' 'shoulder_abduction' 'knee_extension'
 'sitting_hip_flexion' 'curl_up' 'sit_leg_swing' 'calf_raise'
 'knee_flexion' 'hip_extension' 'walking' 'sit_to_stand'
 'knee_extension_static' 'bridge_dynamic' 'dont_know' 'bridge_static'
 'lying_triceps_extension' 'longsitting_hip_flexion'
 'hooklying_knee_extension']
